In [11]:
# Please execute/shift-return this cell everytime you run the notebook.  Don't edit it. 
%load_ext autoreload
%autoreload 2
from notebook import * 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Case study: matrix multiplications

GEMM that computes C = A $\times$ B is the core of many AI/ML applications. The most naive implementation of GEMM takes $O(n^3)$. Assume it takes 1 second to perform GEMM on 1,024$\times$1,024$\times$1,024 matrices. How much time do you expect it would take for 2,048$\times$2,048$\times$2,048 matrices?

In [12]:
render_code("matrix_mul/mm.c", show=["//START","//END"])

// matrix_mul/mm.c:54-75 (22 lines)
//START
void mm(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  uint64_t i,j,k;
  for(i = 0; i < M; i++)
  {
    for(j = 0; j < K; j++)
    {
      for(k = 0; k < N; k++)
      {
        c[i][j] += a[i][k]*b[k][j];
        #ifdef DUMP
          fprintf(stderr, "a[%ld][%ld], %p\n",i,k, &a[i][k]);
          fprintf(stderr, "b[%ld][%ld], %p\n",k,j, &b[k][j]);
          fprintf(stderr, "c[%ld][%ld], %p\n",i,j, &c[i][j]);
        #endif
      }
    }
  }
  return;
}
//END

In [13]:
! cd matrix_mul; make clean; make mm

rm -f blockmm mm blockmm_transpose cachegrind.* mm_dump rect_blockmm_trans blockmm_transpose_reg blockmm_reg
gcc -DHAVE_LINUX_PERF_EVENT_H -O3 mm.c perfstats.c -o mm 


In [14]:
! cd matrix_mul; echo "size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > mm.csv
for size in (512, 1024, 2048, 4096):
    ! ./matrix_mul/mm {size} >> ./matrix_mul/mm.csv

234410496.000000,1406510080.000000,10521102336.000000,48070299648.000000,

In [16]:
display_df_mono(render_csv("matrix_mul/mm.csv"))

,index,size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,512,1055345676,725437770,0.687394,0.186749,0.135475,0.240146,126484891,526699450
1,1,1024,8613829844,11433457382,1.327337,0.179260,2.049566,0.232245,998788845,4300582600
2,2,2048,69007256610,123208944010,1.785449,0.179111,22.068129,0.310622,10703302500,34457667827
3,3,4096,553031805372,1530510999378,2.767492,0.179115,274.137870,0.519088,143295918429,276053384241


In [17]:
print(2.087376/0.132256)
print(21.879449/2.087376)
print(274.137870/22.068129)

15.782845390757316
10.481795804876555
12.422343099408202


WOW! Compuational complexty breaks again! The GEMM performance go wild because of cache misses!

What kind of misses are we seeing?

In [18]:
! make -C matrix_mul mm_dump; ./matrix_mul/mm_dump 256 >& mm_dump_address.csv

make: Entering directory '/nfshome/htseng/courses/CS203/demo/matrix_mul/matrix_mul'
gcc -DHAVE_LINUX_PERF_EVENT_H -DDUMP -O3 mm.c perfstats.c -o mm_dump 
make: Leaving directory '/nfshome/htseng/courses/CS203/demo/matrix_mul/matrix_mul'


In [19]:
! echo "element,address" > mm_dump_addresses_digest.csv 
! head -n 101 mm_dump_address.csv | grep "b\[" >> mm_dump_addresses_digest.csv
df = pd.read_csv("mm_dump_addresses_digest.csv",skipfooter=1,engine='python')
df["address"] = df["address"].str.replace('0x','')
df["address"]=df[["address"]].apply(lambda x: x.astype(str).map(lambda x: int(x, base=16)))
# only show the first N addresses 
#N = 32
#df2 = df2.iloc[:N]
C = 49152
B = 64
A = 12
offset_bits = int(math.log2(B))
S = int(C/(B*A))
index_bits = int(math.log2(S))
df["tag"]=(df["address"].apply(lambda x: x >> (offset_bits+index_bits)))
df["tag"] = df["tag"].apply(lambda x: hex(x))
df["index"] = df["address"].apply(lambda x: hex((x>>offset_bits)%S))
df["address"] = df["address"].apply(lambda x: hex(x))
display_df_mono(df)

,element,address,tag,index
0,b[0][0],0x7c00d34ff000,0x7c00d34ff,0x0
1,b[1][0],0x7c00d34ff800,0x7c00d34ff,0x20
2,b[2][0],0x7c00d3500000,0x7c00d3500,0x0
3,b[3][0],0x7c00d3500800,0x7c00d3500,0x20
4,b[4][0],0x7c00d3501000,0x7c00d3501,0x0
5,b[5][0],0x7c00d3501800,0x7c00d3501,0x20
6,b[6][0],0x7c00d3502000,0x7c00d3502,0x0
7,b[7][0],0x7c00d3502800,0x7c00d3502,0x20
8,b[8][0],0x7c00d3503000,0x7c00d3503,0x0
9,b[9][0],0x7c00d3503800,0x7c00d3503,0x20
